In [1]:
# ==========================================
# IMPORTS
# ==========================================
import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State

import pandas as pd
import numpy as np
import plotly.graph_objs as go

from pickle import load
from scipy.optimize import minimize


# ==========================================
# LOAD DATA
# ==========================================
investors = pd.read_csv('/Users/shardone/Desktop/SW/금융 전략을 위한 머신러닝/InputData.csv', index_col=0)
assets = pd.read_csv('/Users/shardone/Desktop/SW/금융 전략을 위한 머신러닝/SP500Data.csv', index_col=0)

missing_fractions = assets.isnull().mean().sort_values(ascending=False)
drop_list = sorted(list(missing_fractions[missing_fractions > 0.3].index))
assets.drop(labels=drop_list, axis=1, inplace=True)
assets = assets.ffill()

options = [{'label': tic, 'value': tic} for tic in assets.columns]


# ==========================================
# DASH APP
# ==========================================
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets)

app.layout = html.Div([

    html.H2("Robo Advisor Dashboard"),

    html.H4("Investor Characteristics"),

    html.Label("Age"),
    dcc.Slider(id='Age', min=20, max=80, step=1, value=40),

    html.Label("Education"),
    dcc.Slider(id='Edu', min=1, max=4, step=1, value=2),

    html.Label("Married"),
    dcc.Slider(id='Married', min=0, max=1, step=1, value=0),

    html.Label("Kids"),
    dcc.Slider(id='Kids', min=0, max=5, step=1, value=1),

    html.Label("Occupation"),
    dcc.Slider(id='Occ', min=1, max=4, step=1, value=2),

    html.Label("Income"),
    dcc.Slider(id='Inccl', min=1, max=8, step=1, value=4),

    html.Label("Risk Preference"),
    dcc.Slider(id='Risk', min=1, max=4, step=1, value=2),

    html.Label("Net Worth"),
    dcc.Slider(id='Nwcat', min=1, max=10, step=1, value=5),

    html.Button("Predict Risk Tolerance", id='investor_char_button'),

    html.Br(), html.Br(),

    dcc.Input(id='risk-tolerance-text', type='text', readOnly=True),

    html.Hr(),

    html.H4("Select Assets"),

    dcc.Dropdown(
        id='ticker_symbol',
        options=options,
        multi=True
    ),

    html.Button("Get Allocation", id='submit-asset_alloc_button'),

    dcc.Graph(id='Asset-Allocation'),
    dcc.Graph(id='Performance')

], style={
    'backgroundColor': '#f9f9f9',
    'color': '#111111',
    'padding': '40px'
})


# ==========================================
# RISK PREDICTION
# ==========================================
def predict_riskTolerance(X_input):
    filename = '/Users/shardone/Desktop/SW/금융 전략을 위한 머신러닝/finalized_model.sav'
    loaded_model = load(open(filename, 'rb'))
    feature_names = loaded_model.feature_names_in_
    X_df = pd.DataFrame(X_input, columns=feature_names)
    predictions = loaded_model.predict(X_df)
    return predictions


# ==========================================
# SHARPE RATIO OPTIMIZATION
# ==========================================
def get_asset_allocation(riskTolerance, stock_ticker):

    assets_selected = assets.loc[:, stock_ticker]
    returns = assets_selected.pct_change().dropna()

    mu = returns.mean().values
    Sigma = returns.cov().values
    n = len(mu)

    # Sharpe Ratio (riskTolerance 반영)
    def negative_sharpe(weights):
        port_return = np.dot(mu, weights)
        port_vol = np.sqrt(np.dot(weights.T, np.dot(Sigma, weights)))

        if port_vol == 0:
            return 0

        # riskTolerance 높을수록 위험 허용 증가
        adjusted_return = port_return * (1 + riskTolerance)

        return -(adjusted_return / port_vol)

    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    bounds = tuple((0, 1) for _ in range(n))
    init_guess = np.ones(n) / n

    result = minimize(
        negative_sharpe,
        init_guess,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )

    if not result.success:
        print("Optimization failed:", result.message)
        weights = init_guess
    else:
        weights = result.x

    Alloc = pd.DataFrame(
        weights,
        index=assets_selected.columns,
        columns=['Weight']
    )

    portfolio_returns = np.dot(returns.values, weights)
    portfolio_cum = (1 + portfolio_returns).cumprod() * 100

    return Alloc, pd.DataFrame(portfolio_cum, index=returns.index)


# ==========================================
# CALLBACKS
# ==========================================
@app.callback(
    Output('risk-tolerance-text', 'value'),
    Input('investor_char_button', 'n_clicks'),
    State('Age', 'value'),
    State('Edu', 'value'),
    State('Married', 'value'),
    State('Kids', 'value'),
    State('Occ', 'value'),
    State('Inccl', 'value'),
    State('Risk', 'value'),
    State('Nwcat', 'value')
)
def update_risk_tolerance(n_clicks, Age, Edu, Married, Kids, Occ, Inccl, Risk, Nwcat):

    if n_clicks is None:
        return ''

    X_input = [[Age, Edu, Married, Kids, Occ, Inccl, Risk, Nwcat]]
    val = predict_riskTolerance(X_input)[0]

    return str(round(float(val * 100), 2))


@app.callback(
    Output('Asset-Allocation', 'figure'),
    Output('Performance', 'figure'),
    Input('submit-asset_alloc_button', 'n_clicks'),
    State('risk-tolerance-text', 'value'),
    State('ticker_symbol', 'value')
)
def update_asset_allocationChart(n_clicks, risk_tolerance, stock_ticker):

    if not risk_tolerance or not stock_ticker:
        return {}, {}

    risk_val = float(risk_tolerance) / 100

    Alloc, perf = get_asset_allocation(risk_val, stock_ticker)

    fig_alloc = {
        'data': [
            go.Bar(
                x=Alloc.index,
                y=Alloc['Weight']
            )
        ],
        'layout': {'title': "Optimal Asset Allocation (Sharpe Max)"}
    }

    fig_perf = {
        'data': [
            go.Scatter(
                x=perf.index,
                y=perf.iloc[:, 0]
            )
        ],
        'layout': {'title': "Portfolio Value of $100 Investment"}
    }

    return fig_alloc, fig_perf


# ==========================================
# RUN SERVER
# ==========================================
if __name__ == '__main__':
    app.run_server(debug=True)